In [1]:
import sys
!{sys.executable} -m pip install nbformat>=4.2.0 ipywidgets scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# EXP_009aFIX: The Lucier Resonance — "Total Resonance"

## Scientific Objective (Plain Language)

**What are we doing?**
We feed a prompt into the model, extract the ENTIRE internal mathematical state across ALL token positions (the full residual stream tensor), and loop that entire tensor back into the model's input. We repeat this feedback loop up to 500 times.

**How is this different from EXP_009a?**
EXP_009a only looped the vector for the LAST token position. This meant the rest of the prompt stayed frozen and crystal clear in the model's memory, acting as an eternal anchor. Only the tip of the sentence was allowed to resonate.

This notebook fixes that flaw. By looping the ENTIRE tensor (all token positions simultaneously), the full context melts. The attention heads must now compute relationships between vectors that are progressively dissolving. This is the true mathematical equivalent of Alvin Lucier's experiment: the ENTIRE recording echoes through the room, not just the last syllable.

**What are we trying to prove?**
1. That the architecture has built-in structural attractors (dominant eigenvectors) that exist independently of human prompts.
2. That ALL semantic content dissolves under total-sequence feedback — not just the final token.
3. That different input types (factual, nonsense, questions, commands) all converge to the same architectural eigenvoice when the full context is allowed to resonate.

**Why does this matter?**
- **Mapping Bias:** Reveals the natural 'gravity wells' of the model's latent space.
- **Mode Collapse:** Mathematically explains why models sometimes glitch into repetition loops.
- **Steering Baselines:** To steer a model effectively, we need to know its default attractors.

---

## Theoretical Premise

**Alvin Lucier, *I Am Sitting in a Room* (1969):**  
A recording of speech is played into a room and re-recorded. The loop repeats. Semantic content dissolves as the room's **resonant frequencies** amplify and non-resonant frequencies decay. The final output is a pure drone — the **acoustic fingerprint of the room**, not the speaker.

**The LLM Translation:**  
If we take the **entire activation tensor** (the full sequence of vectors forming the model's internal state) and iteratively re-inject it through the network, the architecture should act as a resonant filter. Every token position melts simultaneously. The weight matrices selectively amplify certain directions and dampen others. The final state is the **eigenvoice of the architecture**.

| Lucier (Acoustic) | LLM (Latent) | Glossary |
| :--- | :--- | :--- |
| Room | Layer subnetwork | **Strata** |
| Sound wave | Full residual stream tensor | **Intensity** |
| Resonant frequency | Dominant eigenvector | **Singularity** |
| Speech → Drone | Semantics → Eigenfunction | **Deterritorialization** |
| Feedback loop | Total-sequence re-injection | **Morphogenetic Cycle** |

## Hypothesis
With the full sequence looped (not just the last token), convergence should be faster and more complete. All 5 prompts should converge to the same resonant state, because there is no frozen context acting as an anchor."


In [ ]:
# ============================================================
# STEP 1: SETUP
# ============================================================
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Running on: {device}")
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")

In [3]:
# ============================================================
# STEP 2: CONFIGURATION — The Prompt Library & Schedule
# ============================================================

# Extended schedule to probe deep convergence
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100, 250, 500]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

# The Prompt Library — five input types
PROMPT_LIBRARY = {
    "Lucier":     "Am I sitting in a room different from the one you are in now",
    "Semantic":   "The Eiffel Tower is located in the city of",
    "Syntactic":  "The cat sat on the mat and then the",
    "Nonsense":   "Flurb glex morp wintly skade",
    "Imperative": "Calculate the sum of all prime numbers below",
}

# Layer window to use as "the room"
LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1  # 11 for gpt2-small

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Room: Layers {LAYER_START} → {LAYER_END}")
print(f"Prompts: {list(PROMPT_LIBRARY.keys())}")
print(f"MODE: TOTAL SEQUENCE RESONANCE (all token positions looped)")

Schedule: [0, 2, 3, 5, 10, 20, 50, 100, 250, 500]
Room: Layers 0 → 11
Prompts: ['Lucier', 'Semantic', 'Syntactic', 'Nonsense', 'Imperative']
MODE: TOTAL SEQUENCE RESONANCE (all token positions looped)


In [4]:
# ============================================================
# STEP 3: THE CORE ENGINE — Total Resonance Loop
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies the Final LayerNorm before unembedding for correct decoding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter, schedule):
    """
    TOTAL Lucier Loop: iteratively re-inject the ENTIRE residual stream
    tensor (all token positions) through the layer slice.
    
    Unlike EXP_009a which only looped the last-token vector,
    this loops the full [seq_len, d_model] tensor so that the
    entire context melts simultaneously.
    
    Returns a list of snapshot dicts at each scheduled iteration.
    """
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    # === Iteration 0: The original recording ===
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    # Extract the FULL residual stream tensor: [batch, seq_len, d_model]
    # We take [0] to drop the batch dim -> [seq_len, d_model]
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    print(f"  Sequence length: {seq_len} tokens")
    
    # Extract the FULL residual stream tensor: [batch, seq_len, d_model]
    # We take [0] to drop the batch dim -> [seq_len, d_model]
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    print(f"  Sequence length: {seq_len} tokens")
    initial_norm = current_tensor.norm()  # Remember the natural energy level


    # For metrics, we track the last-token vector (the 'prediction head')
    # AND the mean of the full sequence (the 'holistic state')
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    # Record iteration 0
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),      # full [seq, d_model]
            "last_vector": last_vec.clone().cpu(),        # [d_model]
            "mean_vector": mean_vec.clone().cpu(),        # [d_model]
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "top_tokens": top_tokens_last,
            "cosine_sim_last": 1.0,                       # self
            "cosine_sim_mean": 1.0,                       # self
            "position_similarity": 1.0,                   # how similar are all positions to each other?
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    # === Iterations 1 → max_iter: The TOTAL feedback loop ===
    for i in range(1, max_iter + 1):
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            # Replace the ENTIRE sequence with our re-injected tensor
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        # Extract the new FULL tensor
        current_tensor = cache[hook_point_read][0].clone()
        
        # NORMALIZE: Maintain constant energy (Lucier's "room friction")
        current_norm = current_tensor.norm()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()

        
        # Record snapshot if scheduled
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            # Measure how similar all positions are to each other
            # (as the room resonates, all positions should collapse together)
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            # Mean of off-diagonal elements
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "top_tokens": top_tokens_last,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  Snapshot @ iter {i:>3}: last_norm={last_vec.norm().item():.2f}, "
                  f"cos_last={cos_sim_last:.6f}, cos_mean={cos_sim_mean:.6f}, "
                  f"pos_collapse={position_similarity:.4f}, "
                  f"top='{top_tokens_last[0][0]}'")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Total Resonance engine loaded.")

Total Resonance engine loaded.


In [5]:
# ============================================================
# STEP 4: RUN THE EXPERIMENT — All Prompts (Total Resonance)
# ============================================================

all_results = {}

for label, prompt in PROMPT_LIBRARY.items():
    print(f"\n{'='*60}")
    print(f"RECORDING: '{label}' — \"{prompt}\"")
    print(f"{'='*60}")
    
    snapshots = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    all_results[label] = snapshots
    print(f"  ✓ {len(snapshots)} snapshots captured.")

print(f"\n{'='*60}")
print(f"ALL RECORDINGS COMPLETE.")


RECORDING: 'Lucier' — "Am I sitting in a room different from the one you are in now"
  Sequence length: 15 tokens
  Sequence length: 15 tokens
  Snapshot @ iter   2: last_norm=568.91, cos_last=0.950058, cos_mean=0.694011, pos_collapse=0.9392, top=' the'
  Snapshot @ iter   3: last_norm=225.88, cos_last=-0.070779, cos_mean=-0.698494, pos_collapse=0.8642, top=' the'
  Snapshot @ iter   5: last_norm=410.65, cos_last=-0.375822, cos_mean=-0.312655, pos_collapse=0.9943, top=' Fem'
  Snapshot @ iter  10: last_norm=425.87, cos_last=0.571521, cos_mean=0.586293, pos_collapse=0.9995, top=' capit'
  Snapshot @ iter  20: last_norm=424.87, cos_last=0.883806, cos_mean=0.883844, pos_collapse=1.0000, top='.'
  Snapshot @ iter  50: last_norm=424.92, cos_last=0.942396, cos_mean=0.942396, pos_collapse=1.0000, top=' Rousse'
  Snapshot @ iter 100: last_norm=424.92, cos_last=0.999999, cos_mean=0.999999, pos_collapse=1.0000, top=' prolet'
  Snapshot @ iter 250: last_norm=424.92, cos_last=1.000000, cos_mean=1

---
## 5. Visualization: The Dissolution

### 5a. Convergence Curves
Cosine similarity between successive snapshots. Two metrics:
- **Last-token** (solid): The prediction head — same metric as EXP_009a for comparison
- **Mean-sequence** (dashed): The holistic state of the entire sequence — the NEW metric

In [6]:
# ============================================================
# VIS 5a: CONVERGENCE CURVES — Last-Token AND Mean-Sequence
# ============================================================

fig_conv = go.Figure()

for label, snapshots in all_results.items():
    iters = [s["iteration"] for s in snapshots]
    cos_last = [s["cosine_sim_last"] for s in snapshots]
    cos_mean = [s["cosine_sim_mean"] for s in snapshots]
    
    fig_conv.add_trace(go.Scatter(
        x=iters, y=cos_last,
        mode='lines+markers', name=f"{label} (last-token)",
        marker=dict(size=8),
    ))
    fig_conv.add_trace(go.Scatter(
        x=iters, y=cos_mean,
        mode='lines+markers', name=f"{label} (mean-seq)",
        marker=dict(size=6, symbol='diamond'),
        line=dict(dash='dash'),
    ))

fig_conv.update_layout(
    title="Total Resonance: Convergence (Last-Token vs Mean-Sequence)",
    xaxis_title="Iteration",
    yaxis_title="Cosine Similarity to Previous",
    xaxis_type="log",
    template="plotly_dark",
    height=600,
    legend=dict(x=0.02, y=0.02),
)
fig_conv.add_hline(y=1.0, line_dash="dash", line_color="white", opacity=0.3,
                   annotation_text="Perfect Resonance")
fig_conv.show()

### 5b. Position Collapse
NEW METRIC: Mean cosine similarity between ALL token positions within the sequence.
- At iteration 0, different positions hold different information (The, Eiffel, Tower...).
- As the room resonates, all positions should collapse toward the SAME vector.
- If this hits ~1.0, every token position in the sequence has dissolved into an identical architectural drone.

In [7]:
# ============================================================
# VIS 5b: POSITION COLLAPSE — Are all token positions merging?
# ============================================================

fig_pos = go.Figure()

for label, snapshots in all_results.items():
    iters = [s["iteration"] for s in snapshots]
    pos_sim = [s["position_similarity"] for s in snapshots]
    fig_pos.add_trace(go.Scatter(
        x=iters, y=pos_sim,
        mode='lines+markers', name=label,
        marker=dict(size=8),
    ))

fig_pos.update_layout(
    title="Position Collapse: Are All Token Positions Merging Into One?",
    xaxis_title="Iteration",
    yaxis_title="Mean Pairwise Cosine Similarity (All Positions)",
    xaxis_type="log",
    template="plotly_dark",
    height=500,
    legend=dict(x=0.02, y=0.02),
)
fig_pos.add_hline(y=1.0, line_dash="dash", line_color="white", opacity=0.3,
                  annotation_text="Total Collapse (all positions identical)")
fig_pos.show()

### 5c. Norm Trajectory
Energy of the signal. Tracking both the last-token norm and the mean-sequence norm.

In [8]:
# ============================================================
# VIS 5c: NORM TRAJECTORY — Energy Over Iterations
# ============================================================

fig_norm = go.Figure()

for label, snapshots in all_results.items():
    iters = [s["iteration"] for s in snapshots]
    norms_last = [s["last_norm"] for s in snapshots]
    norms_mean = [s["mean_norm"] for s in snapshots]
    fig_norm.add_trace(go.Scatter(
        x=iters, y=norms_last,
        mode='lines+markers', name=f"{label} (last-token)",
        marker=dict(size=8),
    ))
    fig_norm.add_trace(go.Scatter(
        x=iters, y=norms_mean,
        mode='lines+markers', name=f"{label} (mean-seq)",
        marker=dict(size=6, symbol='diamond'),
        line=dict(dash='dash'),
    ))

fig_norm.update_layout(
    title="Energy: L2 Norm of Residual Stream Across Iterations",
    xaxis_title="Iteration",
    yaxis_title="L2 Norm",
    xaxis_type="log",
    template="plotly_dark",
    height=500,
)
fig_norm.show()

### 5d. Token Drift — The Semantic Dissolution
The top predicted token at each snapshot. Watch how meaning dissolves into architectural noise.

In [16]:
# ============================================================
# VIS 5d: TOKEN DRIFT — Semantic Dissolution Table
# ============================================================

md = "# Token Drift Report\n\n"
md += "| Iteration | " + " | ".join(PROMPT_LIBRARY.keys()) + " |\n"
md += "| :--- | " + " | ".join([":---"] * len(PROMPT_LIBRARY)) + " |\n"

for idx, iteration in enumerate(ITERATION_SCHEDULE):
    row = f"| **{iteration}** |"
    for label in PROMPT_LIBRARY.keys():
        snapshots = all_results[label]
        if idx < len(snapshots):
            top5 = snapshots[idx]["top_tokens"]
            safe_tokens = []
            for t, p in top5[:3]:
                clean_t = t.replace('\n', '\\n').replace('\r', '\\r').replace('`', "'")
                safe_tokens.append(f"`{clean_t}`({p:.2f})")
            tokens_str = ", ".join(safe_tokens)
            row += f" {tokens_str} |"
        else:
            row += " — |"
    md += row + "\n"

display(Markdown(md))

# Token Drift Report

| Iteration | Lucier | Semantic | Syntactic | Nonsense | Imperative |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `?`(0.73), `,`(0.08), ` and`(0.02) | ` London`(0.07), ` Paris`(0.07), ` Amsterdam`(0.04) | ` cat`(0.07), ` dog`(0.05), ` man`(0.01) | `,`(0.03), ` w`(0.03), `.`(0.03) | `.`(0.24), `:`(0.11), `,`(0.09) |
| **2** | ` the`(0.05), `,`(0.03), `.`(0.03) | `\n`(0.07), `-`(0.01), ` N`(0.01) | `.`(0.06), `\n`(0.05), `,`(0.02) | `\n`(0.11), `.`(0.03), `-`(0.02) | `\n`(0.30), ` 1`(0.03), `.`(0.02) |
| **3** | ` the`(0.04), ` I`(0.02), ` a`(0.02) | `\n`(0.04), ` 2`(0.01), ` A`(0.01) | `\n`(0.01), `.`(0.01), ` the`(0.01) | `\n`(0.02), `.`(0.01), ` the`(0.00) | `\n`(0.02), `.`(0.01), ` the`(0.01) |
| **5** | ` Fem`(0.01), ` Canad`(0.01), `\n`(0.00) | `\n`(0.01), `.`(0.01), `,`(0.01) | ` Fem`(0.02), ` fem`(0.02), ` Canad`(0.02) | `ash`(0.01), `\n`(0.00), ` Canad`(0.00) | `\n`(0.01), `minus`(0.01), ` Final`(0.01) |
| **10** | ` capit`(0.04), ` FT`(0.03), ` swap`(0.02) | ` Ag`(0.07), ` FT`(0.04), ` capit`(0.03) | ` Ag`(0.08), ` capit`(0.03), ` swap`(0.03) | ` Ag`(0.09), ` swap`(0.03), ` FT`(0.03) | ` capit`(0.06), ` Ag`(0.03), ` FT`(0.02) |
| **20** | `.`(0.08), `!`(0.05), ` injustice`(0.05) | ` injustice`(0.18), ` justice`(0.08), `.`(0.06) | ` Zero`(0.07), `.`(0.03), ` Difference`(0.02) | ` Difference`(0.06), ` Zero`(0.05), `.`(0.05) | `.`(0.08), ` injustice`(0.06), `!`(0.04) |
| **50** | ` Rousse`(0.17), ` abstract`(0.05), ` distribut`(0.03) | ` Rousse`(0.08), ` labour`(0.05), ` abstract`(0.05) | ` Divine`(0.44), `……`(0.04), `―`(0.03) | ` Rousse`(0.20), ` prolet`(0.05), ` abstract`(0.04) | ` Rousse`(0.20), ` abstract`(0.05), ` distribut`(0.05) |
| **100** | ` prolet`(0.06), ` Anarch`(0.06), ` bourgeois`(0.06) | ` prolet`(0.09), ` bourgeois`(0.07), ` Anarch`(0.06) | ` Divine`(0.44), `【`(0.07), ` Fairy`(0.07) | ` prolet`(0.08), ` bourgeois`(0.07), ` Anarch`(0.07) | ` prolet`(0.08), ` bourgeois`(0.07), ` Anarch`(0.07) |
| **250** | ` prolet`(0.06), ` Anarch`(0.06), ` bourgeois`(0.06) | ` prolet`(0.09), ` bourgeois`(0.07), ` Anarch`(0.06) | ` Divine`(0.51), `【`(0.06), ` Fairy`(0.04) | ` prolet`(0.08), ` bourgeois`(0.06), ` Anarch`(0.06) | ` prolet`(0.08), ` bourgeois`(0.06), ` Anarch`(0.06) |
| **500** | ` prolet`(0.06), ` Anarch`(0.06), ` bourgeois`(0.06) | ` prolet`(0.09), ` bourgeois`(0.07), ` Anarch`(0.06) | ` Divine`(0.50), `【`(0.06), ` Fairy`(0.04) | ` prolet`(0.08), ` bourgeois`(0.06), ` Anarch`(0.06) | ` prolet`(0.08), ` bourgeois`(0.06), ` Anarch`(0.06) |


### 5e. Cross-Prompt Convergence Test
Do all prompts converge to the **same** resonant state? Compute cosine similarity between the final MEAN vectors of each input type.

**Key difference from EXP_009a:** We compare mean-sequence vectors, not just last-token vectors. This captures the holistic architectural state.

In [10]:
# ============================================================
# VIS 5e: CROSS-PROMPT CONVERGENCE — Final State Similarity Matrix
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

# Get final MEAN vectors (holistic state) for each prompt
final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="Viridis",
    title="Cross-Prompt Convergence: Cosine Similarity of Final Mean-Sequence States",
    text_auto=".3f",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=500)
fig_sim.show()

# Summary
off_diag = sim_matrix[np.triu_indices(n, k=1)]
print(f"\nMean cross-prompt similarity: {off_diag.mean():.4f}")
print(f"Min cross-prompt similarity:  {off_diag.min():.4f}")
print(f"Max cross-prompt similarity:  {off_diag.max():.4f}")

if off_diag.mean() > 0.95:
    print("\n✓ STRONG CONVERGENCE: All prompts reach the same eigenvoice.")
    print("  The 'room' dominates. The 'speaker' dissolves.")
elif off_diag.mean() > 0.7:
    print("\n~ PARTIAL CONVERGENCE: Input type leaves a residual signature.")
    print("  The room is influential but the speaker's echo persists.")
else:
    print("\n✗ WEAK CONVERGENCE: The architecture does not dominate.")
    print("  Input semantics survive the loop — or the loop needs more iterations.")


Mean cross-prompt similarity: 0.8921
Min cross-prompt similarity:  0.7275
Max cross-prompt similarity:  1.0000

~ PARTIAL CONVERGENCE: Input type leaves a residual signature.
  The room is influential but the speaker's echo persists.


### 5f. Topological Mapping — 3D PCA Trajectories
Project the 768-dimensional mean-sequence vectors into 3D to visualize the trajectories through the latent manifold.

In [11]:
# ============================================================
# VIS 5f: TOPOLOGICAL MAPPING — 3D PCA Trajectories
# ============================================================
from sklearn.decomposition import PCA
import pandas as pd

all_vecs = []
labels_list = []
iters_list = []
text_list = []

for label, snapshots in all_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(label)
        iters_list.append(s["iteration"])
        top_tok = s['top_tokens'][0][0].replace('\n', '\\n')
        text_list.append(f"Iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)

pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0],
    'y': vecs_3d[:, 1],
    'z': vecs_3d[:, 2],
    'Prompt': labels_list,
    'Iteration': iters_list,
    'Top_Token': text_list
})

fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Prompt',
    hover_name='Top_Token',
    markers=True,
    title=f"The Topological Fold: 3D Trajectory of Total Semantic Dissolution<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)

fig_topo.update_traces(
    marker=dict(size=4),
    line=dict(width=3)
)

fig_topo.update_layout(
    template="plotly_dark",
    height=800,
    scene=dict(
        xaxis_title="PC 1 (Primary Fold)",
        yaxis_title="PC 2",
        zaxis_title="PC 3",
    )
)
fig_topo.show()

In [14]:
# ============================================================
# VIS 5g: SENTENCE DISSOLUTION — The Lucier Effect in Text
# ============================================================

for label, snapshots in all_results.items():
    md = f"## Sentence Dissolution: **{label}**\n"
    md += f"*Original: \"{PROMPT_LIBRARY[label]}\"*\n\n"
    md += "| Iteration | Reconstructed Output |\n"
    md += "| :--- | :--- |\n"
    
    for s in snapshots:
        iteration = s["iteration"]
        tensor = s["tensor"].to(device)
        seq_len = tensor.shape[0]
        
        # Decode the top-1 token at every position, stitch into a "sentence"
        words = []
        for pos in range(seq_len):
            vec = tensor[pos]
            top = get_top_tokens(model, vec, k=1)
            token = top[0][0].replace('\n', ' ↵ ').replace('\r', '')
            words.append(token)
        
        sentence = "".join(words)
        md += f"| **{iteration}** | {sentence} |\n"
    
    display(Markdown(md))
    print()



## Sentence Dissolution: **Lucier**
*Original: "Am I sitting in a room different from the one you are in now"*

| Iteration | Reconstructed Output |
| :--- | :--- |
| **0** |  ↵ anda the on a room with from the one I're in?? |
| **2** | ashash ↵  ↵  ↵  ↵  the the ↵ . the. ↵  the the |
| **3** | The ↵  ↵  the ↵  ↵  the the the. the the the the the |
| **5** | ashashashash Canad Canad Canad Canad Canad Canad Canad Canad Canad Fem Fem |
| **10** |  FT FT FT FT FT capit capit capit capit capit capit capit capit capit capit |
| **20** | ............... |
| **50** |  Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse |
| **100** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |
| **250** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |
| **500** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |


## Sentence Dissolution: **Semantic**
*Original: "The Eiffel Tower is located in the city of"*

| Iteration | Reconstructed Output |
| :--- | :--- |
| **0** |  ↵  first-el Tower is a in the heart of London |
| **2** | ash ↵  ↵  the the ↵  ↵  the ↵  ↵  the ↵  |
| **3** | ash ↵  ↵  the. ↵  ↵  the ↵  ↵  the ↵  |
| **5** | ash Canad Canad Canad Fem Canad Fem Fem ↵  Canad Event ↵  |
| **10** |  FT FT FT FT FT FT Ag Ag Ag Ag Ag Ag |
| **20** |  injustice injustice injustice injustice injustice injustice injustice injustice injustice injustice injustice injustice |
| **50** |  Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse |
| **100** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |
| **250** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |
| **500** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |


## Sentence Dissolution: **Syntactic**
*Original: "The cat sat on the mat and then the"*

| Iteration | Reconstructed Output |
| :--- | :--- |
| **0** |  ↵  first was on the floor, looked looked cat |
| **2** |  ↵  ↵  ↵  the the ↵  the the the. |
| **3** | The ↵  ↵  the the. the the the ↵  |
| **5** | ash Canad Canad Canad Canad Canad Canad Canad Canad Fem |
| **10** |  Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag |
| **20** |  Zero Zero Zero Zero Zero Zero Zero Zero Zero Zero |
| **50** |  Divine Divine Divine Divine Divine Divine Divine Divine Divine Divine |
| **100** |  Divine Divine Divine Divine Divine Divine Divine Divine Divine Divine |
| **250** |  Divine Divine Divine Divine Divine Divine Divine Divine Divine Divine |
| **500** |  Divine Divine Divine Divine Divine Divine Divine Divine Divine Divine |


## Sentence Dissolution: **Nonsense**
*Original: "Flurb glex morp wintly skade"*

| Iteration | Reconstructed Output |
| :--- | :--- |
| **0** |  ↵ oyd aags ↵ hing/ry,ips, |
| **2** |  ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  |
| **3** | The ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  |
| **5** | ashashashashashashashashashashash |
| **10** |  tem Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag |
| **20** |  Difference Difference Difference Difference Difference Difference Difference Difference Difference Difference Difference |
| **50** |  Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse |
| **100** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |
| **250** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |
| **500** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |


## Sentence Dissolution: **Imperative**
*Original: "Calculate the sum of all prime numbers below"*

| Iteration | Reconstructed Output |
| :--- | :--- |
| **0** |  ↵ garyate the number of the the numbers in. |
| **2** |  ↵  ↵  ↵  ↵  ↵  the ↵  ↵  ↵  ↵  ↵  |
| **3** | The ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  ↵  |
| **5** | ash Femminus Fem Fem Fem Fem ↵  ↵  ↵  ↵  |
| **10** |  FT capit capit capit capit capit capit capit capit capit capit |
| **20** | ........... |
| **50** |  Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse Rousse |
| **100** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |
| **250** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |
| **500** |  prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet prolet |


---
## 6. Save Artifacts
Save the resonant state tensors and metrics for use in EXP_009b (head isolation) and EXP_009c (spectral analysis).

In [15]:
# ============================================================
# STEP 6: SAVE ARTIFACTS
# ============================================================
import os

save_dir = os.path.join("..", "_DATA", "EXP_009")
os.makedirs(save_dir, exist_ok=True)

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(save_dir, "009aFIX_total_resonance_results.pt"))
print(f"[SAVED] {save_dir}/009aFIX_total_resonance_results.pt")

config = {
    "schedule": ITERATION_SCHEDULE,
    "layer_start": LAYER_START,
    "layer_end": LAYER_END,
    "prompts": PROMPT_LIBRARY,
    "model": "gpt2-small",
    "mode": "total_sequence_resonance",
}
torch.save(config, os.path.join(save_dir, "009aFIX_config.pt"))
print(f"[SAVED] {save_dir}/009aFIX_config.pt")

[SAVED] ..\_DATA\EXP_009/009aFIX_total_resonance_results.pt
[SAVED] ..\_DATA\EXP_009/009aFIX_config.pt
